In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,LabelEncoder

In [2]:
df = pd.read_csv("Ames Housing Dataset.csv")

In [3]:
df.shape

(1460, 81)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [6]:
df.drop(columns=['Id'], inplace = True)

In [7]:
df.duplicated().sum()

0

In [8]:
df.isnull().sum().sort_values(ascending=False) 

PoolQC         1453
MiscFeature    1406
Alley          1369
Fence          1179
MasVnrType      872
               ... 
Heating           0
HeatingQC         0
MSZoning          0
1stFlrSF          0
SalePrice         0
Length: 80, dtype: int64

In [9]:
nans = [col for col in df.columns if df[col].isnull().sum() > 0]

In [10]:
nans

['LotFrontage',
 'Alley',
 'MasVnrType',
 'MasVnrArea',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Electrical',
 'FireplaceQu',
 'GarageType',
 'GarageYrBlt',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PoolQC',
 'Fence',
 'MiscFeature']

In [11]:
for col in nans:
    print(f"{col} nulls :",df[col].isnull().sum())

LotFrontage nulls : 259
Alley nulls : 1369
MasVnrType nulls : 872
MasVnrArea nulls : 8
BsmtQual nulls : 37
BsmtCond nulls : 37
BsmtExposure nulls : 38
BsmtFinType1 nulls : 37
BsmtFinType2 nulls : 38
Electrical nulls : 1
FireplaceQu nulls : 690
GarageType nulls : 81
GarageYrBlt nulls : 81
GarageFinish nulls : 81
GarageQual nulls : 81
GarageCond nulls : 81
PoolQC nulls : 1453
Fence nulls : 1179
MiscFeature nulls : 1406


In [12]:
for col in nans:
    if df[col].isnull().sum() > 600:
        df.drop(columns = [col], inplace = True)
        

In [13]:
nans = [col for col in df.columns if df[col].isnull().sum() > 0 ]

In [18]:
for col in nans:
    print(f"{col} nulls :",df[col].isnull().sum())
    print(f"Data type: {df[col].dtype}")


LotFrontage nulls : 259
Data type: float64
MasVnrArea nulls : 8
Data type: float64
BsmtQual nulls : 37
Data type: object
BsmtCond nulls : 37
Data type: object
BsmtExposure nulls : 38
Data type: object
BsmtFinType1 nulls : 37
Data type: object
BsmtFinType2 nulls : 38
Data type: object
Electrical nulls : 1
Data type: object
GarageType nulls : 81
Data type: object
GarageYrBlt nulls : 81
Data type: float64
GarageFinish nulls : 81
Data type: object
GarageQual nulls : 81
Data type: object
GarageCond nulls : 81
Data type: object


In [15]:
df[nans]

,LotFrontage,MasVnrArea,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Electrical,GarageType,GarageYrBlt,GarageFinish,GarageQual,GarageCond
0,65.0,196.0,Gd,TA,No,GLQ,Unf,SBrkr,Attchd,2003.0,RFn,TA,TA
1,80.0,0.0,Gd,TA,Gd,ALQ,Unf,SBrkr,Attchd,1976.0,RFn,TA,TA
2,68.0,162.0,Gd,TA,Mn,GLQ,Unf,SBrkr,Attchd,2001.0,RFn,TA,TA
3,60.0,0.0,TA,Gd,No,ALQ,Unf,SBrkr,Detchd,1998.0,Unf,TA,TA
4,84.0,350.0,Gd,TA,Av,GLQ,Unf,SBrkr,Attchd,2000.0,RFn,TA,TA
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,62.0,0.0,Gd,TA,No,Unf,Unf,SBrkr,Attchd,1999.0,RFn,TA,TA
1456,85.0,119.0,Gd,TA,No,ALQ,Rec,SBrkr,Attchd,1978.0,Unf,TA,TA
1457,66.0,0.0,TA,Gd,No,GLQ,Unf,SBrkr,Attchd,1941.0,RFn,TA,TA
1458,68.0,0.0,TA,TA,Mn,GLQ,Rec,FuseA,Attchd,1950.0,Unf,TA,TA


In [16]:
df.isnull().any(axis = 1).sum()

366

In [20]:
imputer = SimpleImputer(missing_values=np.nan,strategy='mean')

In [ ]:
df[df[nans].select_dtypes("number").columns] = imputer.fit_transform(df[nans].select_dtypes("number"))

In [24]:
df[df[nans].select_dtypes("number").columns].isnull().sum()

LotFrontage    0
MasVnrArea     0
GarageYrBlt    0
dtype: int64

In [25]:
imputer = SimpleImputer(missing_values=np.nan,strategy='most_frequent')

In [26]:
df[df[nans].select_dtypes("O").columns] = imputer.fit_transform(df[nans].select_dtypes("O"))

In [27]:
df[df[nans].select_dtypes("O").columns].isnull().sum()

BsmtQual        0
BsmtCond        0
BsmtExposure    0
BsmtFinType1    0
BsmtFinType2    0
Electrical      0
GarageType      0
GarageFinish    0
GarageQual      0
GarageCond      0
dtype: int64

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

target = "SalePrice"
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer([
    ("numeric", make_pipeline(
        StandardScaler()
    ), numeric_cols),
    ("categorical", make_pipeline(
        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    ), categorical_cols)
])

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [29]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression , Lasso ,Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.experimental import enable_hist_gradient_boosting  # Needed for HistGB
from sklearn.ensemble import (RandomForestRegressor,ExtraTreesRegressor,BaggingRegressor,
                              AdaBoostRegressor ,GradientBoostingRegressor,HistGradientBoostingRegressor
                              , VotingRegressor,StackingRegressor)
#import xgboost as xgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
#import lightgbm as lgb
from lightgbm import LGBMRegressor


models ={
    "Linear Regression":LinearRegression(),
    "Lasso":Lasso(),
    "Ridge":Ridge(),
    "KNN":KNeighborsRegressor(n_neighbors=5,metric='minkowski',p=1), #  minkowski , euclidean , manhattan
    "SVM":SVR(kernel='linear', C=100, gamma=1), # kernel='rbf' / kernel='poly', degree=4 / kernel='sigmoid
    "Decision Tree":DecisionTreeRegressor(criterion='squared_error',max_depth=10), #{"squared_error", "friedman_mse", "absolute_error", "poisson"}
    "Random Forest":RandomForestRegressor(n_estimators=50,criterion='squared_error',max_depth=10), #{"squared_error", "friedman_mse", "absolute_error", "poisson"}
    "ExtraTrees": ExtraTreesRegressor(n_estimators=100, random_state=42),
    "Bagging": BaggingRegressor(estimator=DecisionTreeRegressor(criterion='squared_error',max_depth=10),n_estimators=50, random_state=42),
    "AdaBoost": AdaBoostRegressor(n_estimators=15, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=100, learning_rate=0.1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0),
    "CatBoost": CatBoostRegressor(verbose=0, iterations=100, learning_rate=0.1, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mse = mean_squared_error(y_test, predictions)

    results[name] = {
        "MAE": mean_absolute_error(y_test, predictions),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_test, predictions),
    }

results_df = pd.DataFrame.from_dict(results, orient="index")
results_df.index.name = "Model"

# Show the best RMSE first
results_df = results_df.sort_values("RMSE")
results_df

e:\anaconda\Lib\site-packages\sklearn\experimental\enable_hist_gradient_boosting.py:18: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(
e:\anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "e:\anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "e:\anaconda\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
    

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001227 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3113
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 68
[LightGBM] [Info] Start training from score 181441.541952


,MAE,MSE,RMSE,R2
Model,,,,
XGBoost,16463.585938,6.459167e+08,25414.892012,0.915790
CatBoost,17307.628208,7.424744e+08,27248.383616,0.903202
GradientBoosting,17119.390329,8.033543e+08,28343.505815,0.895265
ExtraTrees,17513.970890,8.139468e+08,28529.752278,0.893884
LightGBM,16358.285814,8.368146e+08,28927.747556,0.890902
Bagging,18087.555603,8.440688e+08,29052.862564,0.889957
Random Forest,17905.609602,8.470960e+08,29104.914009,0.889562
HistGradientBoosting,16724.080513,8.679551e+08,29461.077048,0.886842
Linear Regression,21483.070233,1.182321e+09,34384.900695,0.845858
